# Fooocus_extend – VSW Google Colab

Diese Version ist für die einfache Nutzung in Schulungen und Workshops vorbereitet.

## Start
1. Oben auf **▶** bei **„Fooocus_extend STARTEN / NEUSTARTEN“** klicken.
2. Beim **allerersten Start** werden benötigte Paketversionen angepasst und Colab startet die Laufzeit einmal automatisch neu.
3. Nach der automatischen Wiederverbindung dieselbe Start-Zelle **noch einmal** ausführen.
4. Den angezeigten `gradio.live`-Link öffnen.

> **Die Runtime 2025.07 und eine T4-GPU sind bereits im Notebook hinterlegt.** Eine manuelle Auswahl über „Laufzeit → Laufzeittyp ändern“ ist im Normalfall nicht nötig.

## Datenhaltung
- **Notebook:** dauerhaft auf GitHub bzw. optional als Kopie in Google Drive.
- **Programmumgebung:** temporär unter `/content` in der Colab-VM.
- **Bilder:** standardmäßig temporär. Nur bei `GoogleDrive_output = True` werden Ergebnisse dauerhaft nach `MyDrive/outputs` geschrieben.

**Sicherheit:** Google Drive nur bei Bedarf verbinden. Keine Passwörter, Tokens, API-Keys oder vertraulichen Inhalte in öffentliche Notebook-Ausgaben schreiben.

Nach der Übung: **Laufzeit → Laufzeit trennen und löschen**.


In [ ]:
# @title ▶ Fooocus_extend STARTEN / NEUSTARTEN

import os
import sys
import shutil
import subprocess
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

# ============================================================
# EINSTELLUNGEN
# ============================================================

Fooocus_Profile = "realistic" #@param ["default", "realistic", "anime"]
Fooocus_Theme = "dark" #@param ["dark", "light"]
Tunnel = "gradio" #@param ["gradio", "cloudflared"]
Memory_patch = True #@param {type:"boolean"}
GoogleDrive_output = False #@param {type:"boolean"}

REPO_URL = "https://github.com/shaitanzx/Fooocus_extend.git"
REPO_DIR = Path("/content/Fooocus_extend")
OUTPUT_DIR = Path("/content/drive/MyDrive/outputs")
PORT = "7865"

# ============================================================
# 1. RUNTIME PRÜFEN
# ============================================================

print(f"Python: {sys.version.split()[0]}")

if sys.version_info >= (3, 13):
    raise RuntimeError(
        "Diese Sitzung verwendet unerwartet Python 3.13 oder neuer.\n"
        "Das Notebook ist auf Colab Runtime 2025.07 (Python 3.11) festgelegt.\n\n"
        "Bitte die Seite neu laden. Falls Colab die hinterlegte Runtime nicht übernimmt:\n"
        "Laufzeit → Laufzeittyp ändern → Runtime-Version 2025.07 und GPU/T4 auswählen."
    )

gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True
)

if gpu_check.returncode != 0:
    raise RuntimeError(
        "Keine NVIDIA-GPU erkannt.\n"
        "Das Notebook fordert automatisch eine T4-GPU an. Falls Colab aktuell keine GPU zuweist:\n"
        "Laufzeit → Laufzeittyp ändern → GPU/T4 auswählen."
    )

print("✓ GPU:", gpu_check.stdout.strip())

# ============================================================
# 2. ABHÄNGIGKEITEN PRÜFEN
# ============================================================

required_packages = {
    "nvidia-cudnn-cu12": "9.1.0.70",
    "pygit2": "1.15.1",
    "numpy": "1.26.4",
}

def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None

to_install = []

for package, wanted_version in required_packages.items():
    current = installed_version(package)
    if current != wanted_version:
        print(f"{package}: {current or 'nicht installiert'} → {wanted_version}")
        to_install.append(f"{package}=={wanted_version}")

if to_install:
    print("\nEinmalige Vorbereitung: benötigte Pakete werden angepasst ...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *to_install],
        text=True,
        capture_output=True
    )

    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            "Die benötigten Paketversionen konnten nicht installiert werden.\n"
            "Bitte prüfen, ob tatsächlich die im Notebook hinterlegte Runtime 2025.07 verwendet wird."
        )

    print("\n✓ Vorbereitung abgeschlossen.")
    print("Colab startet die Laufzeit jetzt einmal automatisch neu.")
    print("Nach der Wiederverbindung bitte dieselbe START-Zelle noch einmal ausführen.")

    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)

else:
    print("✓ Abhängigkeiten stimmen.")

    # ========================================================
    # 3. CUDA PRÜFEN
    # ========================================================

    import torch

    if not torch.cuda.is_available():
        raise RuntimeError(
            "Die GPU ist vorhanden, aber PyTorch erkennt CUDA nicht.\n"
            "Bitte die Laufzeit einmal neu starten und diese Zelle erneut ausführen."
        )

    print("✓ CUDA:", torch.cuda.get_device_name(0))

    # ========================================================
    # 4. ALTE PROZESSE BEENDEN
    # ========================================================

    subprocess.run(
        ["pkill", "-f", "entry_with_update.py"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    subprocess.run(
        ["pkill", "-f", "cloudflared tunnel"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    # ========================================================
    # 5. FOOOCUS_EXTEND INSTALLIEREN / AKTUALISIEREN
    # ========================================================

    if not REPO_DIR.exists():
        print("\nFooocus_extend wird installiert ...")
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    else:
        if not (REPO_DIR / ".git").exists():
            raise RuntimeError(
                "/content/Fooocus_extend existiert, ist aber kein gültiges Git-Repository."
            )

        print("\nFooocus_extend ist bereits vorhanden – Programmcode wird aktualisiert.")
        subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"])
        subprocess.check_call(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"])

    print("✓ Fooocus_extend ist aktuell.")

    # ========================================================
    # 6. GOOGLE DRIVE OPTIONAL
    # ========================================================

    output_args = []

    if GoogleDrive_output:
        from google.colab import drive

        if not Path("/content/drive/MyDrive").exists():
            print("\nGoogle Drive wird verbunden ...")
            drive.mount("/content/drive", force_remount=False)
        else:
            print("✓ Google Drive ist bereits verbunden.")

        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        output_args = ["--output-path", str(OUTPUT_DIR)]
        print("✓ Ausgabeordner:", OUTPUT_DIR)

    # ========================================================
    # 7. CLOUDFLARED OPTIONAL
    # ========================================================

    if Tunnel == "cloudflared":
        print("\nCloudflared wird vorbereitet ...")

        if shutil.which("cloudflared") is None:
            deb_file = "/tmp/cloudflared-linux-amd64.deb"
            subprocess.check_call([
                "wget", "-q", "-O", deb_file,
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"
            ])
            subprocess.check_call(["dpkg", "-i", deb_file])

        subprocess.check_call([
            sys.executable,
            str(REPO_DIR / "patcher_tunel.py")
        ])

    # ========================================================
    # 8. START
    # ========================================================

    args = [
        sys.executable,
        "entry_with_update.py",
        "--port", PORT
    ]

    if Fooocus_Profile == "realistic":
        args += ["--preset", "realistic"]
    elif Fooocus_Profile == "anime":
        args += ["--preset", "anime"]

    if Fooocus_Theme == "dark":
        args += ["--theme", "dark"]

    if Tunnel == "gradio":
        args += ["--share"]

    if Memory_patch:
        args += ["--always-high-vram", "--all-in-fp16"]

    args += output_args

    os.chdir(REPO_DIR)

    print("\n" + "=" * 60)
    print("FOOOCUS_EXTEND WIRD GESTARTET")
    print("=" * 60)
    print("Profil:              ", Fooocus_Profile)
    print("Theme:               ", Fooocus_Theme)
    print("Tunnel:              ", Tunnel)
    print("Memory Patch:        ", Memory_patch)
    print("Google Drive Output: ", GoogleDrive_output)
    print("GPU:                 ", torch.cuda.get_device_name(0))
    print("=" * 60 + "\n")

    subprocess.run(args, check=True)


## Hinweise

- Das Notebook ist auf **Colab Runtime 2025.07** und **GPU/T4** festgelegt.
- Colab-Ressourcen sind trotzdem nicht garantiert. Falls keine GPU verfügbar ist, kann Colab eine manuelle Auswahl oder einen späteren Versuch verlangen.
- `GoogleDrive_output = False` ist bewusst der Standard.
- Nach der Übung: **Laufzeit → Laufzeit trennen und löschen**.
